# 🤖 Multi-AI Problem Solver (Gradio UI)

This notebook provides a complete Gradio web application embedded directly in the notebook.

It uses multiple AI models to analyze a problem, generate diverse solutions using different models (`nvidia/nemotron-3-super-120b-a12b` and `qwen/qwen3.7-flash`), and synthesize the results.

Run the cell below to load the UI!

In [ ]:
import os
import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Defaulting to OpenRouter since the requested model formats (e.g. org/model) 
# strongly suggest an OpenRouter or similar endpoint.
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
API_KEY = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def analyzer_agent(problem: str, model: str, num_solutions: int = 2) -> list:
    """Agent 1: Breaks down the problem into different approaches."""
    prompt = (
        f"You are an expert problem analyzer. The user has a problem: '{problem}'.\n"
        f"Come up with {num_solutions} completely distinct approaches to solve this problem.\n"
        "Return the result as a JSON object with a single key 'approaches' containing a list of strings, "
        "where each string describes one unique approach."
    )
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant designed to output JSON."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"} 
        )
        result = json.loads(response.choices[0].message.content)
        return result.get("approaches", [])
    except Exception as e:
        return [f"Fallback Approach 1 (Error parsing JSON: {e})", "Fallback Approach 2"]

def solver_agent(problem: str, approach: str, model: str) -> str:
    """Agent 2: Solves the problem using a specific approach."""
    prompt = (
        f"You are an expert problem solver.\n"
        f"Problem: {problem}\n"
        f"Approach to use: {approach}\n"
        "Provide a detailed, step-by-step solution to the problem using ONLY the specified approach."
    )
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are an expert problem solver."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error generating solution: {str(e)}"

def synthesize_agent(problem: str, solutions: list, model: str) -> str:
    """Agent 3: Summarizes all the findings."""
    solutions_text = "\n\n".join([f"Solution {i+1}:\n{sol}" for i, sol in enumerate(solutions)])
    prompt = (
        f"Problem: {problem}\n\n"
        f"Here are multiple solutions generated by different models and approaches:\n{solutions_text}\n\n"
        "Provide a brief executive summary comparing these solutions, highlighting the pros and cons of each."
    )
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are an expert at synthesizing information."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error during synthesis: {str(e)}"

def solve_problem(problem: str, analyzer_model: str, solver_models: list, num_solutions: int):
    if not problem.strip():
        yield "Please enter a problem.", ""
        return
        
    output_log = f"### 🎯 Problem: {problem}\n\n"
    yield output_log, ""
    
    output_log += f"**🔍 Analyzer Agent ({analyzer_model})** is brainstorming approaches...\n\n"
    yield output_log, ""
    
    approaches = analyzer_agent(problem, analyzer_model, int(num_solutions))
    
    for i, app in enumerate(approaches):
        output_log += f"**Approach {i+1}:** {app}\n\n"
    yield output_log, ""
    
    all_solutions = []
    
    for i, approach in enumerate(approaches):
        output_log += f"---\n### Developing Approach {i+1}\n\n"
        yield output_log, ""
        
        for model in solver_models:
            output_log += f"**💡 Solver Agent ({model})** is working on it...\n\n"
            yield output_log, ""
            
            solution = solver_agent(problem, approach, model)
            all_solutions.append(f"Model: {model} | Approach: {approach}\nSolution:\n{solution}")
            
            output_log += f"<details><summary>View Solution from {model}</summary>\n\n{solution}\n\n</details>\n\n"
            yield output_log, ""
            
    output_log += "---\n### 📝 Synthesizer Agent is summarizing...\n\n"
    yield output_log, ""
    
    synthesis = synthesize_agent(problem, all_solutions, analyzer_model)
    
    yield output_log, synthesis

def build_ui():
    models = ["nvidia/nemotron-3-super-120b-a12b", "qwen/qwen3.7-flash"]
    
    with gr.Blocks(title="Multi-AI Problem Solver", theme=gr.themes.Soft()) as app:
        gr.Markdown("# 🤖 Multi-AI Problem Solver")
        gr.Markdown("This system uses multiple AI models to analyze a problem, generate diverse solutions using different models, and synthesize the results.")
        
        with gr.Row():
            with gr.Column(scale=1):
                problem_input = gr.Textbox(
                    label="What is your problem?", 
                    lines=4, 
                    placeholder="e.g. How can we effectively scale our microservices database architecture?"
                )
                
                with gr.Group():
                    gr.Markdown("### Agent Configuration")
                    analyzer_dropdown = gr.Dropdown(
                        choices=models, 
                        value=models[1], 
                        label="Analyzer & Synthesizer Model"
                    )
                    solver_checkboxes = gr.CheckboxGroup(
                        choices=models, 
                        value=models, 
                        label="Solver Models (Generates a solution for each approach)"
                    )
                    num_approaches = gr.Slider(minimum=1, maximum=5, value=2, step=1, label="Number of Approaches")
                
                submit_btn = gr.Button("🚀 Generate Solutions", variant="primary")
                
            with gr.Column(scale=2):
                log_output = gr.Markdown(label="Execution Log")
                final_synthesis = gr.Markdown(label="Final Synthesis")
                
        submit_btn.click(
            solve_problem,
            inputs=[problem_input, analyzer_dropdown, solver_checkboxes, num_approaches],
            outputs=[log_output, final_synthesis]
        )
        
    return app

# Launch the UI inside the notebook
app = build_ui()
app.launch(inline=True, inbrowser=False)